In [2]:
%pip install ucimlrepo


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import Libraries
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt
import seaborn as sns

# Set pandas display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)        # Auto-detect width
pd.set_option('display.max_colwidth', 50)   # Limit column width

#importing dataset by ID   
adult = fetch_ucirepo(id=2)

# Verify data was fetched successfully 
#referance https://github.com/uci-ml-repo/ucimlrepo/blob/main/src/demo.ipynb
if adult.data is not None:
    X = adult.data.features
    y = adult.data.targets
else:
    raise ValueError("Failed to fetch dataset.")

# Combine for easier analysis
df = pd.concat([X, y], axis=1)

# Dataset overview
print("\n" + "=" * 80)
print("1. DATASET OVERVIEW")
display(df.head(10))

# Identify column data types 
#Reference:https://www.geeksforgeeks.org/data-analysis/data-cleansing-introduction/ 
cat_col = [col for col in df.columns if df[col].dtype == 'object']
num_col = [col for col in df.columns if df[col].dtype != 'object']

print(f"\nColumn Data Types:")
print(f"Categorical columns ({len(cat_col)}): {cat_col}")
print(f"Numerical columns ({len(num_col)}): {num_col}")

# Summary 
print("Summary")
print(f"Total samples: {df.shape[0]:,}")
print(f"Total attributes: {df.shape[1]}")




1. DATASET OVERVIEW


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
5,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K
6,49,Private,160187,9th,5,Married-spouse-absent,Other-service,Not-in-family,Black,Female,0,0,16,Jamaica,<=50K
7,52,Self-emp-not-inc,209642,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,45,United-States,>50K
8,31,Private,45781,Masters,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084,0,50,United-States,>50K
9,42,Private,159449,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,5178,0,40,United-States,>50K



Column Data Types:
Categorical columns (9): ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']
Numerical columns (6): ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Summary
Total samples: 48,842
Total attributes: 15


## 2. Data Cleaning/Preparation

### Pandas Methods Used:

- df.drop() - Remove columns
- df.replace() - Replace values
- df.dropna() - Remove missing values
- df.str.strip() - Remove whitespace
- df.isnull() - Check for NaN
- df.value_counts() - Count unique values
- df.describe() - Summary statistics
- df.duplicated() - Check for duplicate rows
- df.drop_duplicates() - Remove duplicate rows

In [2]:
# Checking the data before cleaning
# Check for missing values (marked as '?')
print("\nMissing values check per feature:")

# Convert to list to ensure it's iterable
object_columns = list(df.select_dtypes(include=['object']).columns)

for col in object_columns:
    missing_count = (df[col] == '?').sum()
    if missing_count > 0:
        missing_pct = round((missing_count / df.shape[0]) * 100, 2)
        print(f"{col}: {missing_count:,} ({missing_pct}%)")


Missing values check per feature:
workclass: 1,836 (3.76%)
occupation: 1,843 (3.77%)
native-country: 583 (1.19%)


In [3]:
# Removing irrelevant columns
# Define columns to remove. // We can modify this list based on our product.
features_to_remove = ['fnlwgt', 'education-num']

# drop columns and replace '?' with NaN
df_cleaned = df.drop(columns=features_to_remove, errors='ignore').replace('?', np.nan)

# Store initial row count for comparison
rows_before = len(df_cleaned)

# Remove rows with any missing values
df_cleaned = df_cleaned.dropna()
# Remove duplicate rows
duplicates_before = df_cleaned.duplicated().sum()
df_cleaned = df_cleaned.drop_duplicates()

# Calculate cleaning statistics
rows_after = len(df_cleaned)
rows_removed = rows_before - rows_after
data_retention = (rows_after / rows_before) * 100
# Display cleaning summary
print(f"\nColumns removed: {features_to_remove}")
print(f"Rows with missing values removed: {rows_before - len(df_cleaned) - duplicates_before:,}")
print(f"Duplicate rows removed: {duplicates_before:,}")
print(f"Total rows removed: {rows_removed:,} ({100 - data_retention:.2f}%)")
print(f"Data retention: {data_retention:.2f}%")
print(f"Cleaned dataset: {df_cleaned.shape[0]:,} rows, {df_cleaned.shape[1]} features")



Columns removed: ['fnlwgt', 'education-num']
Rows with missing values removed: 3,620
Duplicate rows removed: 4,188
Total rows removed: 7,808 (15.99%)
Data retention: 84.01%
Cleaned dataset: 41,034 rows, 13 features


In [ ]:
# If one of our targets is income, let's standardize it
# Standardize income column (remove whitespace and trailing periods)
#df_cleaned['income'] = df_cleaned['income'].str.strip().str.rstrip('.')

#print("Income distribution:")
#print(df_cleaned['income'].value_counts())
#print(f" Income categories: {df_cleaned['income'].nunique()}")


Columns that were removed: ['education-num', 'fnlwgt']

Reasons:
  - education-num: Duplicate of 'education' column (numerical encoding)
  - fnlwgt: Census sampling weigh


Future cleaning for a specific target such as age, gender,education...

Identification: For numerical columns like 'age', 'fnlwgt', 'capital-gain', 'capital-loss', and 'hours-per-week', we will need to identify extreme values that might be data entry errors or genuine but unusual observations 
. For example, an 'age' of 0 or an 'hours-per-week' of 1000 would be considered an outlier.


2. Standardizing Categorical Data
Categorical variables often contain inconsistencies that need standardization.

Inconsistent Entries: Check for variations in spelling, capitalization, or different representations of the same category (e.g., "United-States" vs. "United States" or "HS-grad" vs. "HS Grad"). The provided data consistently uses "United-States" and "HS-grad"